# ☁️ Voice Cloning — Backend on Colab GPU

Tomar laptop e NVIDIA GPU nai, tai **backend ta Colab er free GPU te** chalabo. Eta ekta **public URL** dibe — oi URL frontend er `.env` (`VITE_API_URL`) e boshale tomar local React app Colab er GPU use korbe.

**Age koro:** `Runtime` → `Change runtime type` → **T4 GPU** → Save.

### Steps
1. GPU check
2. Backend files upload (`main.py`, `engines.py`, `requirements.txt` — `webapp/backend/` folder theke)
3. Install
4. Run + public URL

In [ ]:
import torch
assert torch.cuda.is_available(), '⚠️ GPU nai! Runtime > Change runtime type > T4 GPU koro.'
print('GPU:', torch.cuda.get_device_name(0))

## 2. Backend files upload
Run korle file-picker ashbe. Tomar computer er `webapp/backend/` folder theke **teen ta file** select koro: `main.py`, `engines.py`, `requirements.txt`.

In [ ]:
from google.colab import files
print('main.py, engines.py, requirements.txt upload koro...')
up = files.upload()
print('\nUploaded:', list(up.keys()))

## 3. Install dependencies (~5-8 min)

In [ ]:
!pip -q install -r requirements.txt
print('Installed ✓')

## 4. Run backend + public URL
Ei cell backend chalu kore ekta `trycloudflare.com` URL dey. **Oi URL copy koro** ebong frontend er `.env` e `VITE_API_URL=` er por boshao, tarpor frontend restart (`npm run dev`).

> Ei cell **chalu rakhte hobe** — bondho korle backend bondho. Prothom `/api/clone` call e model load hobe (kichu somoy).

In [ ]:
import subprocess, time, re, threading

# uvicorn background e
server = subprocess.Popen(['uvicorn', 'main:app', '--host', '0.0.0.0', '--port', '8000'])
print('Backend starting... 8 sec wait')
time.sleep(8)

# cloudflared tunnel
!wget -q -O /content/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x /content/cloudflared
tunnel = subprocess.Popen(['/content/cloudflared', 'tunnel', '--url', 'http://localhost:8000'],
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
url = None
for line in tunnel.stdout:
    m = re.search(r'https://[-a-z0-9]+\.trycloudflare\.com', line)
    if m:
        url = m.group(0)
        break
print('\n' + '='*60)
print('🔗 BACKEND URL (frontend .env e boshao):')
print('   VITE_API_URL=' + (url or 'FAILED — cell abar run koro'))
print('='*60)
print('\n⚠️  Ei cell chalu thakuk. Test:', (url + '/api/health') if url else '')

In [ ]:
# (Optional) backend log dekhte — server chalu thakle ei cell log dekhabe na;
# error debug korte uporer cell er output dekho, ba nicher health check koro.
import requests
try:
    print(requests.get('http://localhost:8000/api/health').json())
except Exception as e:
    print('health check fail:', e)